In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta

# ==========================================
# 1. DOWNLOAD DATA
# ==========================================
ticker = "CNH=X"  # Yahoo Finance ticker for USD/CNH
interval = "15m"
period = "60d"    # Yahoo Finance limits 15m data to the last 360 days

print(f"Downloading {interval} data for {ticker}...")
df = yf.download(ticker, period=period, interval=interval)

# Flatten MultiIndex columns if using recent versions of yfinance
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.droplevel(1)

# Keep only the necessary columns
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ==========================================
# 2. FEATURE ENGINEERING (Multi-Timeframe)
# ==========================================
print("Calculating technical indicators...")

# --- 15-MINUTE TIMEFRAME FEATURES ---
# 1. Price Action: Log Returns
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# 2. Awesome Oscillator (AO) 15m
ao_15m = ta.momentum.AwesomeOscillatorIndicator(high=df['High'], low=df['Low'], window1=5, window2=34)
df['AO_15m'] = ao_15m.awesome_oscillator()

# 3. Money Flow Index (MFI) - 10, 14, 20 periods
df['MFI_10'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=10).money_flow_index()
df['MFI_14'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=14).money_flow_index()
df['MFI_20'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=20).money_flow_index()

# 4. Volume Ratio (current volume / simple moving average of volume)
sma_volume = df['Volume'].rolling(window=20).mean()
df['Vol_Ratio'] = df['Volume'] / (sma_volume + 1e-8)

# 5. Bollinger Bands Percent B
bb = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['BB_PctB'] = bb.bollinger_pband()

# 6. Donchian Position (20 period)
donchian_high = df['High'].rolling(window=20).max()
donchian_low = df['Low'].rolling(window=20).min()
df['Donchian_Pos_20'] = (df['Close'] - donchian_low) / (donchian_high - donchian_low + 1e-8)

# --- 4-HOUR TIMEFRAME FEATURES ---
print("Resampling to 4H timeframe...")
df_4h = df.resample('4H').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

# ADX and AO for 4H
adx_4h = ta.trend.ADXIndicator(high=df_4h['High'], low=df_4h['Low'], close=df_4h['Close'], window=14)
df_4h['ADX_4H'] = adx_4h.adx()

ao_4h = ta.momentum.AwesomeOscillatorIndicator(high=df_4h['High'], low=df_4h['Low'], window1=5, window2=34)
df_4h['AO_4H'] = ao_4h.awesome_oscillator()

# Forward-fill to align with 15m data
df_4h_reindexed = df_4h[['ADX_4H', 'AO_4H']].reindex(df.index, method='ffill')
df['ADX_4H'] = df_4h_reindexed['ADX_4H']
df['AO_4H'] = df_4h_reindexed['AO_4H']

# --- 1-DAY TIMEFRAME FEATURES ---
print("Resampling to 1D timeframe...")
df_1d = df.resample('1D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

# ADX and AO for 1D
adx_1d = ta.trend.ADXIndicator(high=df_1d['High'], low=df_1d['Low'], close=df_1d['Close'], window=14)
df_1d['ADX_1D'] = adx_1d.adx()

ao_1d = ta.momentum.AwesomeOscillatorIndicator(high=df_1d['High'], low=df_1d['Low'], window1=5, window2=34)
df_1d['AO_1D'] = ao_1d.awesome_oscillator()

# Forward-fill to align with 15m data
df_1d_reindexed = df_1d[['ADX_1D', 'AO_1D']].reindex(df.index, method='ffill')
df['ADX_1D'] = df_1d_reindexed['ADX_1D']
df['AO_1D'] = df_1d_reindexed['AO_1D']

# ==========================================
# 3. CREATE TARGET VARIABLE (Normalized)
# ==========================================
# Calculate the log return of the next candle
df['Next_Log_Return'] = np.log(df['Close'].shift(-1) / df['Close'])

# Normalize to [0, 1] range using rolling standard deviation for adaptive scaling
rolling_std = df['Next_Log_Return'].rolling(window=20).std()
rolling_mean = df['Next_Log_Return'].rolling(window=20).mean()

# Normalize: (value - mean) / (2 * std) + 0.5, clipped to [0, 1]
# This centers 0 change at 0.5, with ±1 std mapped to ~0.16 and ~0.84
df['Target'] = ((df['Next_Log_Return'] - rolling_mean) / (2 * rolling_std) + 0.5).clip(0, 1)

# Drop the helper column
df.drop('Next_Log_Return', axis=1, inplace=True)

# ==========================================
# 4. CLEAN AND SAVE
# ==========================================
# Drop NaN values generated by the lookback periods
df.dropna(inplace=True)

# Save to CSV
filename = "USD_CNH_15m_ML_ready.csv"
df.to_csv(filename)
print(f"Success! {len(df)} rows of data saved to {filename}")

# Also save to USDCNH_MTF_ML_ready.csv
mtf_filename = "USDCNH_MTF_ML_ready.csv"
df.to_csv(mtf_filename)
print(f"Also saved to {mtf_filename}")


[*********************100%***********************]  1 of 1 completed

Calculating technical indicators...
Success! 5623 rows of data saved to USD_CNH_15m_ML_ready.csv


In [2]:
import pandas as pd
import numpy as np
import ta

# ==========================================
# 1. LOAD MT5 DATA
# ==========================================
# Replace with your actual MT5 exported file name
input_file = "USDCNH_M15_202401020000_202604102345.csv"

print(f"Loading data from {input_file}...")
# MT5 exports are tab-separated, so we must specify sep='\t'
df = pd.read_csv(input_file, sep='\t')

# Combine Date and Time into a single Datetime index
df['Datetime'] = pd.to_datetime(df['<DATE>'] + ' ' + df['<TIME>'])
df.set_index('Datetime', inplace=True)

# Rename columns to standard readable names
df = df.rename(columns={
    '<OPEN>': 'Open',
    '<HIGH>': 'High',
    '<LOW>': 'Low',
    '<CLOSE>': 'Close',
    '<TICKVOL>': 'Volume'
})

# Keep only the necessary columns
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# ==========================================
# 2. FEATURE ENGINEERING (Multi-Timeframe)
# ==========================================
print("Calculating technical indicators...")

# --- 15-MINUTE TIMEFRAME FEATURES ---
# 1. Price Action: Log Returns
df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))

# 2. Awesome Oscillator (AO) 15m
ao_15m = ta.momentum.AwesomeOscillatorIndicator(high=df['High'], low=df['Low'], window1=5, window2=34)
df['AO_15m'] = ao_15m.awesome_oscillator()

# 3. Money Flow Index (MFI) - 10, 14, 20 periods
df['MFI_10'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=10).money_flow_index()
df['MFI_14'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=14).money_flow_index()
df['MFI_20'] = ta.volume.MFIIndicator(high=df['High'], low=df['Low'], close=df['Close'], volume=df['Volume'], window=20).money_flow_index()

# 4. Volume Ratio (current volume / simple moving average of volume)
sma_volume = df['Volume'].rolling(window=20).mean()
df['Vol_Ratio'] = df['Volume'] / (sma_volume + 1e-8)

# 5. Bollinger Bands Percent B
bb = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['BB_PctB'] = bb.bollinger_pband()

# 6. Donchian Position (20 period)
donchian_high = df['High'].rolling(window=20).max()
donchian_low = df['Low'].rolling(window=20).min()
df['Donchian_Pos_20'] = (df['Close'] - donchian_low) / (donchian_high - donchian_low + 1e-8)

# --- 4-HOUR TIMEFRAME FEATURES ---
print("Resampling to 4H timeframe...")
df_4h = df.resample('4H').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

# ADX and AO for 4H
adx_4h = ta.trend.ADXIndicator(high=df_4h['High'], low=df_4h['Low'], close=df_4h['Close'], window=14)
df_4h['ADX_4H'] = adx_4h.adx()

ao_4h = ta.momentum.AwesomeOscillatorIndicator(high=df_4h['High'], low=df_4h['Low'], window1=5, window2=34)
df_4h['AO_4H'] = ao_4h.awesome_oscillator()

# Forward-fill to align with 15m data
df_4h_reindexed = df_4h[['ADX_4H', 'AO_4H']].reindex(df.index, method='ffill')
df['ADX_4H'] = df_4h_reindexed['ADX_4H']
df['AO_4H'] = df_4h_reindexed['AO_4H']

# --- 1-DAY TIMEFRAME FEATURES ---
print("Resampling to 1D timeframe...")
df_1d = df.resample('1D').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()

# ADX and AO for 1D
adx_1d = ta.trend.ADXIndicator(high=df_1d['High'], low=df_1d['Low'], close=df_1d['Close'], window=14)
df_1d['ADX_1D'] = adx_1d.adx()

ao_1d = ta.momentum.AwesomeOscillatorIndicator(high=df_1d['High'], low=df_1d['Low'], window1=5, window2=34)
df_1d['AO_1D'] = ao_1d.awesome_oscillator()

# Forward-fill to align with 15m data
df_1d_reindexed = df_1d[['ADX_1D', 'AO_1D']].reindex(df.index, method='ffill')
df['ADX_1D'] = df_1d_reindexed['ADX_1D']
df['AO_1D'] = df_1d_reindexed['AO_1D']

# ==========================================
# 3. CREATE TARGET VARIABLE (Normalized)
# ==========================================
# Calculate the log return of the next candle
df['Next_Log_Return'] = np.log(df['Close'].shift(-1) / df['Close'])

# Normalize to [0, 1] range using rolling standard deviation for adaptive scaling
rolling_std = df['Next_Log_Return'].rolling(window=20).std()
rolling_mean = df['Next_Log_Return'].rolling(window=20).mean()

# Normalize: (value - mean) / (2 * std) + 0.5, clipped to [0, 1]
# This centers 0 change at 0.5, with ±1 std mapped to ~0.16 and ~0.84
df['Target'] = ((df['Next_Log_Return'] - rolling_mean) / (2 * rolling_std) + 0.5).clip(0, 1)

# Drop the helper column
df.drop('Next_Log_Return', axis=1, inplace=True)

# ==========================================
# 4. CLEAN AND SAVE
# ==========================================
# Drop NaN values generated by the lookback periods (e.g., first 26 periods for MACD)
df.dropna(inplace=True)

# Save to CSV
output_file = "USDCNH_15m_ML_ready_2years.csv"
df.to_csv(output_file)
print(f"Success! {len(df)} rows of data saved to {output_file}")

# Also save to USDCNH_MTF_ML_ready.csv
mtf_filename = "USDCNH_MTF_ML_ready.csv"
df.to_csv(mtf_filename)
print(f"Also saved to {mtf_filename}")


Loading data from USDCNH_M15_202401020000_202604102345.csv...
Calculating technical indicators...
Resampling to 4H timeframe...
Resampling to 1D timeframe...


/tmp/ipykernel_35740/343123154.py:64: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_4h = df.resample('4H').agg({


Success! 25820 rows of data saved to USDCNH_15m_ML_ready_2years.csv
Also saved to USDCNH_MTF_ML_ready.csv
